# Day 2, morning: neural models, measured

A **workbench**, like on day 1: you write no code here. Each section calls the package and draws what comes
out. The figures follow lecture L3.

| You implement | In | L3 slides | Section here |
|---|---|---|---|
| `bench_model` | `speech/asr_benchmark.py` | "Measuring neural models" | 4, 5 |

**What "implement" means.** `bench_model` is written except 2 short holes. Each hole is a
`raise NotImplementedError("TODO(d2): ...")` placed where the code goes, and its message says what to write:
replace the `raise` by those lines (keep the indentation) and re-run the cell.

**Plan for the lab (2 h).**

| Time | What | Computer time |
|---|---|---|
| 0:00–0:20 | Sections 1–3 (provided code): LibriSpeech, wav2vec2's frames and labels, Whisper and the normaliser | about 1 min, plus the first download of 3 small models (about 0.5 GB) |
| 0:20–0:45 | Section 4: `bench_model` (2 holes); `pytest -m d2 -k bench` | 1 min |
| 0:45–0:50 | Start the graded run in a terminal (below) | about 11 min |
| 0:50–1:15 | Section 5 (batch 1 vs batch 8) and its questions | 1 min |
| 1:15–1:40 | Section 6: one more model (`--models ... --n 50`), `--plot`, your fronts | 2–5 min |
| 1:40–2:00 | One `DECISIONS.md` entry; commit | |

The graded run (3 models × batch 1 and 16 on 200 utterances):
```
export DSP_MILESTONE=d2          # PowerShell: $env:DSP_MILESTONE = "d2"
python -m speech.asr_benchmark
```

Data: `python scripts/fetch_data.py` puts LibriSpeech test-clean in `data/LibriSpeech` (or set `DSP_DATA`).

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd() if (Path.cwd() / "speech").is_dir() else Path.cwd().parent  # the folder that contains speech/
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # results/ and data/ are relative to the bundle root
DATA = Path(os.environ.get("DSP_DATA", ROOT / "data"))

from IPython.display import Audio, display

from speech.data import load_librispeech, normalise, wer

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
warnings.filterwarnings("ignore")  # transformers and torch are verbose
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def not_yet(e):
    """What a cell prints when it reaches one of your stubs."""
    print(f"Not implemented yet: {e}\nImplement it in the package, then re-run this cell.")

## 1. LibriSpeech test-clean

Read audiobooks, 16 kHz, with their transcripts. `load_librispeech` shuffles the utterances with a fixed seed,
so every group gets the same ones. Here 16 utterances; the graded run uses 200.

In [ ]:
data = load_librispeech(DATA / "LibriSpeech", n=16)
dur = np.array([len(d["audio"]) / d["sr"] for d in data])
print(f"{len(data)} utterances, {dur.sum():.0f} s of audio, from {dur.min():.1f} to {dur.max():.1f} s")
ex = data[0]
print("reference:", ex["text"])
display(Audio(ex["audio"], rate=ex["sr"]))

## 2. wav2vec 2.0: one distribution per 20 ms frame

The CTC model maps the waveform to $T$ frames, one every 320 samples (20 ms), and gives at each frame a
distribution $P_t(k)$ over 32 outputs: the letters, `'`, the word delimiter `|`, the blank (`<pad>`) and 3
special tokens (L3, "wav2vec 2.0: from a waveform to one distribution per frame"). The image shows
$\log P_t(k)$ for the first 3 seconds; the line below it, the most likely label of every frame, before any
collapse: an alignment $\pi$. Collapsing it into text is your `greedy_ctc` of this afternoon.

In [ ]:
import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC

W2V = "facebook/wav2vec2-base-960h"
proc = AutoProcessor.from_pretrained(W2V)
model = Wav2Vec2ForCTC.from_pretrained(W2V).eval()
with torch.inference_mode():
    x = proc(ex["audio"], sampling_rate=ex["sr"], return_tensors="pt").input_values
    logprobs = torch.log_softmax(model(x).logits[0], -1).numpy()  # (T, 32)
vocab = {i: t for t, i in proc.tokenizer.get_vocab().items()}
blank = proc.tokenizer.pad_token_id
T = logprobs.shape[0]
print(f"{len(ex['audio'])} samples -> T = {T} frames (samples / 320 = {len(ex['audio']) / 320:.1f}); "
      f"logprobs shape {logprobs.shape}")

best = logprobs.argmax(-1)
print(f"the blank is the most likely label in {100 * np.mean(best == blank):.0f} % of the frames: CTC outputs are spiky")
shown = min(T, 150)
rows = [blank] + sorted({int(k) for k in best[:shown]} - {blank})
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.imshow(np.maximum(logprobs[:shown, rows].T, -15), aspect="auto", cmap="viridis", interpolation="nearest")
ax.set_yticks(range(len(rows)), ["∅" if k == blank else vocab[k] for k in rows], fontsize=7)
ax.set(xlabel="frame $t$ (20 ms)", title="$\\log P_t(k)$, first frames (rows: the blank and the labels that win somewhere)")
fig.tight_layout()
print("alignment (best label per frame, ∅ = blank):")
print(" ".join("∅" if k == blank else vocab[int(k)] for k in best[:shown]))

**Questions.** How many frames does one letter last when it wins? Where are the word delimiters `|`? Why can
a frame be labelled with a letter although it sees the whole utterance (L3, "Frame vs label")?

## 3. Whisper, and why the WER needs a normaliser

An encoder–decoder writes text one token at a time, with capitals, punctuation and digits. LibriSpeech
references are upper case, without punctuation. `speech.data.wer` normalises both sides first (L3, "Normalise
before you count, and report the normaliser").

In [ ]:
import jiwer
from transformers import pipeline

whisper = pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en", device="cpu")
hyps_w = [o["text"] for o in whisper([{"raw": d["audio"], "sampling_rate": d["sr"]} for d in data[:8]], batch_size=1)]
refs_w = [d["text"] for d in data[:8]]
for r, h in list(zip(refs_w, hyps_w))[:3]:
    print(f"REF {r}\nHYP {h.strip()}\n")
print(f"WER on raw strings: {jiwer.wer(refs_w, hyps_w):.3f}    WER after normalise(): {wer(refs_w, hyps_w):.3f}")
print("normalise:", repr(hyps_w[0].strip()), "->", repr(normalise(hyps_w[0])))

**Question.** What does `normalise` not fix (numbers, "Mr.", British spellings)? Which model families does that
penalise, and how would you report it?

## 4. Your benchmark: `bench_model`

The three default models, batch 1, on the 16 utterances: WER, real-time factor (processing time / audio time),
latency per request (median and 90th percentile), size. The first call of a model downloads it.

In [ ]:
import pandas as pd

from speech.asr_benchmark import CPU_MODELS, bench_model

table, hyps_by_model = [], {}
try:
    for mid in CPU_MODELS:
        m, hyps = bench_model(mid, data, 1, "cpu", False)
        hyps_by_model[mid] = hyps
        table.append(dict(model=mid.split("/")[-1], wer=m["wer"], rtf=m["rtf"], lat_p50_ms=m["lat_p50_ms"],
                          lat_p90_ms=m["lat_p90_ms"], params_M=m["n_params_M"]))
except NotImplementedError as e:
    not_yet(e)
if table:
    display(pd.DataFrame(table).round(3))

Latency against the length of each utterance (batch 1: one request per utterance). The audio length is on the
x axis: real time is the diagonal (RTF = 1).

In [ ]:
from common.bench import time_calls  # provided: run a function on every input, return outputs and seconds

if len(table) == len(CPU_MODELS):
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for mid in CPU_MODELS:
        asr = pipeline("automatic-speech-recognition", model=mid, device="cpu")
        _, lat = time_calls(lambda d: asr({"raw": d["audio"], "sampling_rate": d["sr"]}), data, warmup=1)
        ax.plot(dur, lat, "o", label=mid.split("/")[-1])
    ax.plot([0, dur.max()], [0, dur.max()], "k:", lw=0.8, label="real time")
    ax.set(xlabel="utterance length (s)", ylabel="latency (s)", title="batch 1 on this CPU")
    ax.legend(fontsize=8)
    fig.tight_layout()

**Questions.** Which model is fastest per second of audio, and which is the most accurate? Is either one
dominated? Is the latency proportional to the length for the encoder–decoders, and why not exactly?

## 5. Batch 1 against batch 8: latency or throughput

Batching shares one forward pass between several utterances (padded to the longest one). Below, the same two
models at batch 1 and batch 8. Watch the WER of `wav2vec2-base-960h`: its feature extractor has no attention
mask, so the zero padding changes its output.

In [ ]:
batch_rows = []
try:
    for mid in ("facebook/wav2vec2-base-960h", "openai/whisper-tiny.en"):
        for b in (1, 8):
            m, _ = bench_model(mid, data, b, "cpu", False)
            batch_rows.append(dict(model=mid.split("/")[-1], batch=b, wer=m["wer"], rtf=m["rtf"],
                                   lat_p50_ms=m["lat_p50_ms"]))
except NotImplementedError as e:
    not_yet(e)
if batch_rows:
    display(pd.DataFrame(batch_rows).round(3))

**Questions.** On a CPU, does batching buy throughput (RTF)? What happens to the latency of one request in a
batch? Which batch size would you measure for a push-to-talk assistant, and which for transcribing a podcast
archive?

## 6. Your graded rows, as fronts

After the graded run (top of the notebook) and one more model, e.g.
```
python -m speech.asr_benchmark --models openai/whisper-base.en --n 50
python -m speech.asr_benchmark --plot        # -> results/figures/asr_pareto.png
```
this cell draws the `asr` rows of `results/results.jsonl`: one front per batch size (WER against RTF).
Rows from `--n 50` and from 200 utterances are not on the same test set: say so when you compare them.

In [ ]:
from common.bench import load_records
from common.pareto import plot_pareto

res = ROOT / "results/results.jsonl"
df = load_records(res) if res.exists() else None
if df is None or "task" not in df or (df.task == "asr").sum() == 0:
    print("No 'asr' rows in results/results.jsonl yet: run the graded command above, then re-run this cell.")
else:
    df = df[df.task == "asr"].copy()
    df["cfg_n"] = df["cfg_n"].fillna(0).astype(int) if "cfg_n" in df else 0
    df = df.drop_duplicates(["model", "cfg_batch", "cfg_n"], keep="last")
    df["label"] = df.model.str.split("/").str[-1] + " (n=" + df.cfg_n.astype(str) + ")"
    batches = sorted(df.cfg_batch.unique())[:2]
    fig, axes = plt.subplots(1, len(batches), figsize=(7 * len(batches), 4.5), squeeze=False)
    for ax, b in zip(axes[0], batches):
        plot_pareto(df[df.cfg_batch == b].reset_index(drop=True), "rtf", "wer", "min", "min", label="label",
                    ax=ax, logx=True, title=f"batch {b}")
    fig.tight_layout()

## Before you leave

Write at least one `DECISIONS.md` entry tagged d2 (format in `SUBMISSION_SPEC.md`): for example, which model
you would put behind a push-to-talk button on this laptop, with the rows or the figure that justify it.